# GRAFCET as the description language of a GRAFCET virtual machine (GVM)

This notebook belongs to the paper *Teaching Programmable Logic Controller Programming and Verification with Agent AI:
A Model-First IDEF0–GRAFCET Workflow in a Virtual Factory*. It runs in Google Colab without a PLC or an FPGA.

1. `gvm/grafcet2gvm.py` translates the reference GRAFCET of the tank tasks (`nusmv/tank_reference.json`, Fig. 6 and
   Tables 3–5 of the paper) into a Python program and a C++ program; one call of `tick` is one evolution of the
   GRAFCET to a stable situation.
2. `gvm/e3_check.py` runs both programs in lockstep with the model that NuSMV checks, on every input combination from
   every reachable stable situation and on random input sequences, and counts the ticks in which a program differs
   from the model (E3). Programs with seeded faults must differ.

The ST version of the same program is checked with the OpenPLC compiler (matiec) in `gvm/e3_log.txt`; that check
needs OpenPLC_v3 and is not repeated here.

In [ ]:
!git clone -q https://github.com/neojwang/model-first-plc-agent-ai.git
%cd model-first-plc-agent-ai/gvm

## 1. Generate the programs and check them in lockstep

Expected result: E3 = 0 for Python and C++, and E3 > 0 for every negative control.

In [ ]:
!python3 e3_check.py ../nusmv/tank_reference.json --out /content/gvm_out --seqs 50 --len 50

## 2. Step the Python program by hand

Inputs, in the order of `INPUTS`: start and stop (buttons), mode (timer, tontof, pid), r (level region LOW, MID,
HIGH), and the conditions owned by steps (timer_q, ton_q, tof_q, hold_ok, logged).

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location('tank_gvm', '/content/gvm_out/tank_reference_gvm.py')
gvm = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gvm)
g = gvm.GVM()

def tick(**inputs):
    vec = []
    for name in gvm.INPUTS:
        v = inputs.get(name, 0)
        vec.append(gvm.ENUMS[name].index(v) if isinstance(v, str) else int(v))
    situation, stable, outputs = g.tick(vec)
    print('active steps:', g.active(), '| stable:', stable, '| outputs:', dict(zip(gvm.OUTPUTS, outputs)))

tick(start=1, mode='timer')    # start: macro-step M1 (simple timer), fill valve open
tick(mode='timer')             # filling
tick(mode='timer', timer_q=1)  # timer elapsed: M1 ends, steps 5 and 6 log the data
tick(logged=1)                 # data logged: back to step 0
tick(start=1, mode='pid', stop=1)  # stop has priority: the forcing order of G0 keeps G1 in step 0

## 3. The same ticks with the C++ program

Each output line gives the situation (one bit per step, in the order of `STEP_NAMES`), the stability flag, and the
outputs (fill_valve, pid_on, discharge_at_ud).

In [ ]:
!g++ -std=c++17 -O2 -Wall -o /content/gvm_out/gvm /content/gvm_out/tank_reference_gvm_main.cpp
!printf 'T 1 0 0 0 0 0 0 0 0\nT 0 0 0 0 0 0 0 0 0\nT 0 0 0 0 1 0 0 0 0\nT 0 0 0 0 0 0 0 0 1\nT 1 1 2 0 0 0 0 0 0\n' | /content/gvm_out/gvm